In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.config import COMMUNES, TYPES, 
from src.portal.crawler import run_crawl, RAW_DIR

from concurrent.futures import ThreadPoolExecutor

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
#communes_to_run = {k:COMMUNES[k] for k in ["Concepción", "Talcahuano"]}
communes_to_run = COMMUNES
MAX_PAGES = 50

In [19]:
with ThreadPoolExecutor(max_workers=1) as executor:
    run_dir = executor.submit(run_crawl, communes=communes_to_run, max_pages=MAX_PAGES).result()

print("Guardados en :",run_dir)

Concepción - departamento


Future exception was never retrieved
future: <Future finished exception=BrokenPipeError(32, 'Broken pipe')>
Traceback (most recent call last):
  File "/Users/ianlopezcandia/.pyenv/versions/3.10.0/lib/python3.10/asyncio/unix_events.py", line 675, in write
    n = os.write(self._fileno, data)
BrokenPipeError: [Errno 32] Broken pipe


    page 1: 48 listings
    page 2: 48 listings
    page 3: 48 listings
    page 4: 48 listings
    page 5: 48 listings
    page 6: 48 listings
    page 7: 48 listings
    page 8: 48 listings
    page 9: 48 listings
    page 10: 48 listings
    page 11: 48 listings
    page 12: 48 listings
    page 13: 48 listings
    page 14: 48 listings
    page 15: 48 listings
    page 16: 48 listings
    page 17: 48 listings
    page 18: 48 listings
    page 19: 48 listings
    page 20: 48 listings
    page 21: 24 listings
  -> saved 979 listings

Concepción - casa
    page 1: 48 listings
    page 2: 39 listings
  -> saved 87 listings

Talcahuano - departamento
    page 1: 48 listings
    page 2: 7 listings
  -> saved 55 listings

Talcahuano - casa
    page 1: 30 listings
  -> saved 30 listings

Hualpén - departamento
    page 1: 33 listings
  -> saved 33 listings

Hualpén - casa
    page 1: 2 listings
  -> saved 2 listings

San Pedro de la Paz - departamento
    page 1: 48 listings
    page 2: 48 

In [20]:
RUN = sorted(p.name for p in RAW_DIR.iterdir() if p.is_dir())[-1]
run_dir = RAW_DIR / RUN

files = sorted(run_dir.glob("*.csv"))
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

print(f"Run {RUN}: {len(files)} files, {len(df)} listings")

Run 2026-10-23: 16 files, 1639 listings


In [21]:
df.pivot_table(index="commune", columns="property_type",
               values="id", aggfunc="count", fill_value=0)

property_type,casa,departamento
commune,,
Chiguayante,23,58
Concepción,87,979
Coronel,14,4
Hualpén,2,33
Hualqui,2,0
Penco,1,0
San Pedro de la Paz,61,279
Talcahuano,30,55
Tomé,7,4


In [22]:
#CALIDAD DE LA DATA
pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1),
})

,missing,missing_%
source,0,0.0
id,0,0.0
commune,0,0.0
property_type,0,0.0
title,0,0.0
broker,1409,86.0
price,0,0.0
currency,0,0.0
bedrooms,17,1.0
bathrooms,9,0.5


In [23]:

(df[df["currency"] == "CLP"]
   .groupby(["commune", "property_type"])["price"]
   .median()
   .unstack())

property_type,casa,departamento
commune,,
Chiguayante,1000000.0,610000.0
Concepción,1175000.0,500000.0
Coronel,1200000.0,470000.0
Hualpén,590000.0,450000.0
Hualqui,950000.0,NaN
Penco,500000.0,NaN
San Pedro de la Paz,800000.0,560000.0
Talcahuano,865000.0,580000.0
Tomé,1125000.0,700000.0


In [25]:
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

consolidated = (
    df.drop_duplicates(subset=["id"])
      .drop(columns=["total_m2","broker"])
)

print(f"Rows: {len(df)} -> {len(consolidated)} ({len(df) - len(consolidated)} duplicates removed)")

output_file = PROCESSED_DIR / f"portal_{RUN}.csv"
consolidated.to_csv(output_file, index=False, encoding="utf-8")
print("Saved:", output_file.relative_to(ROOT))

Rows: 1639 -> 1639 (0 duplicates removed)
Saved: data/processed/portal_2026-10-23.csv
